## download (run only once)

In [ ]:
# from pathlib import Path

# DEST = Path("~/GTEx_data").expanduser()
# DEST.mkdir(exist_ok=True)

# META_BASE = "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files"

# meta_files = ["GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
#                 "GTEx_Analysis_v11_Annotations_SubjectPhenotypesDS.txt",
#                 "GTEx_Analysis_v11_Annotations_SampleAttributesDD.xlsx",
#                 "GTEx_Analysis_v11_Annotations_SubjectPhenotypesDD.xlsx",
#                 "GTEx_Analysis_v11_Sample_Tissue_Changes_From_v8.txt"
#               ]

# for f in meta_files:
#     url = f"{META_BASE}/{f}"
#     !wget -c -q --show-progress -P {DEST} {url}

In [ ]:
# MAIN_BASE = "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq"
# main_files = ["GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_reads.parquet",
#               "GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_tpm.parquet"]

# for f in main_files:
#     url = f"{MAIN_BASE}/{f}"
#     !wget -c -q --show-progress -P {DEST} {url}

In [ ]:
# import pandas as pd

# gtf = pd.read_csv("gencode.vXX.GRCh38.genes.gtf", sep="\t", comment="#", header=None,
#                   usecols=[2, 8], names=["feature", "attrs"])
# genes = gtf[gtf["feature"] == "gene"]
# genes = genes.assign(
#     gene_id=genes["attrs"].str.extract(r'gene_id "([^"]+)"', expand=False),
#     gene_type=genes["attrs"].str.extract(r'gene_type "([^"]+)"', expand=False),
# )
# protein_coding = set(genes.loc[genes["gene_type"] == "protein_coding", "gene_id"])

## main processing

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = "~/GTEx_data/GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_tpm.parquet"

In [ ]:
gtex_tpm_pq = pd.read_parquet(DATA_PATH)

In [ ]:
gtex_samp_ids = [v for v in gtex_tpm_pq.columns if v.startswith('GTEX-')]
len(gtex_samp_ids) == len(set(gtex_samp_ids))

In [ ]:
metadata_path = "~/GTEx_data/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
gtex_meta_df_all = pd.read_csv(metadata_path,sep="\t")

In [ ]:
meta_samp_ids = [v for v in gtex_meta_df_all['SAMPID'] if v.startswith('GTEX-')]

In [ ]:
set(gtex_samp_ids) <= set(meta_samp_ids)

In [ ]:
gtex_meta_df = gtex_meta_df_all[gtex_meta_df_all['SAMPID'].isin(gtex_samp_ids)]

In [ ]:
(gtex_meta_df['SMTSD'].value_counts()>100).value_counts()

In [ ]:
broad_fine_tissue_counts = gtex_meta_df.groupby('SMTS')['SMTSD'].value_counts()

In [ ]:
num_fine_per_broad = (broad_fine_tissue_counts>100).groupby('SMTS').sum()

In [ ]:
num_fine_per_broad[num_fine_per_broad>2]

In [ ]:
broad_fine_tissue_counts.loc[num_fine_per_broad[num_fine_per_broad>2].index]

In [ ]:
# # Sample annotations: one row per sample; SMTSD = detailed tissue name
# attrs = pd.read_csv(, sep="\t")
# print(attrs["SMTSD"].value_counts().head(10))

### restrict to brain (for now)

In [ ]:
brain_meta_df = gtex_meta_df[gtex_meta_df['SMTS']=='Brain'].copy()
brain_meta_df.columns

In [ ]:
brain_meta_df['SMTSD_SAMPID'] = brain_meta_df['SMTSD'].str.replace(' ','') + '_' + brain_meta_df['SAMPID']

In [ ]:
sampid_tissue_dict= brain_meta_df.set_index('SAMPID')['SMTSD_SAMPID'].to_dict()

In [ ]:
gtex_tpm_pq = gtex_tpm_pq.set_index('Description')

In [ ]:
gtex_tpm_pq_brain = gtex_tpm_pq[brain_meta_df['SAMPID']].copy()
# del(gtex_tpm_pq)

In [ ]:
# TPM > 0.1 in at least 20% of samples
gtex_tpm_pq_brain = gtex_tpm_pq_brain[((gtex_tpm_pq_brain>0.1).mean(axis=1)>= 0.2)].copy()

In [ ]:
gtex_tpm_pq_brain